In [2]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

PROCESSED = Path("processed")

REQUIRED_FILES = [
    "burninwatch_features_absolute_v2.csv",
    "burninwatch_features_relative_v2.csv",
    "burninwatch_features_residual_v2.csv",
    "burninwatch_features_relationship_v2.csv",
]

missing = [
    f for f in REQUIRED_FILES
    if not (PROCESSED / f).exists()
]

if missing:
    raise FileNotFoundError(
        "Missing required files:\n" +
        "\n".join(f" - {f}" for f in missing)
    )

META = [
    "device_id",
    "file_name",
    "file_role",
    "window_index",
    "window_size",
    "window_start_seconds",
    "window_end_seconds",
]

absolute = pd.read_csv(
    PROCESSED / "burninwatch_features_absolute_v2.csv"
)

relative = pd.read_csv(
    PROCESSED / "burninwatch_features_relative_v2.csv"
)

residual = pd.read_csv(
    PROCESSED / "burninwatch_features_residual_v2.csv"
)

relationship = pd.read_csv(
    PROCESSED / "burninwatch_features_relationship_v2.csv"
)

print("Absolute:", absolute.shape)
print("Relative:", relative.shape)
print("Residual:", residual.shape)
print("Relationship:", relationship.shape)

Absolute: (687, 28)
Relative: (687, 18)
Residual: (687, 15)
Relationship: (687, 12)


In [3]:
# ============================================================
# 1. VALIDATE METADATA KEYS
# ============================================================

for name, df in {
    "absolute": absolute,
    "relative": relative,
    "residual": residual,
    "relationship": relationship,
}.items():

    missing_meta = [
        c for c in META
        if c not in df.columns
    ]

    duplicate_keys = int(
        df.duplicated(META).sum()
    )

    print(
        f"{name:15s} | "
        f"missing metadata = {missing_meta} | "
        f"duplicate keys = {duplicate_keys}"
    )

    if missing_meta:
        raise ValueError(
            f"{name} is missing metadata columns: {missing_meta}"
        )

    if duplicate_keys:
        raise ValueError(
            f"{name} contains duplicate window keys."
        )

absolute        | missing metadata = [] | duplicate keys = 0
relative        | missing metadata = [] | duplicate keys = 0
residual        | missing metadata = [] | duplicate keys = 0
relationship    | missing metadata = [] | duplicate keys = 0


In [4]:
# ============================================================
# 2. IDENTIFY FEATURES EXPLICITLY
# ============================================================

absolute_features = [
    c for c in absolute.columns
    if c not in META
]

relative_features = [
    c for c in relative.columns
    if (
        c.endswith("_delta_from_baseline")
        or c.endswith("_pct_change_from_baseline")
    )
]

residual_features = [
    c for c in residual.columns
    if c not in META
    and c not in {
        "node2Voltage_baseline",
        "collectorEmitterCurrent_baseline",
        "packageTemperature_baseline",
        "expected_current",
        "expected_package_temperature",
    }
]

relationship_all = [
    c for c in relationship.columns
    if c not in META
]

relationship_features = [
    c for c in relationship_all
    if c not in absolute_features
]

print("Absolute:", len(absolute_features))
print("Relative:", len(relative_features))
print("Residual:", len(residual_features))
print("New relationship:", len(relationship_features))

print("\nRelative features:")
for c in relative_features:
    print(" -", c)

print("\nNew relationship features:")
for c in relationship_features:
    print(" -", c)

Absolute: 21
Relative: 6
Residual: 3
New relationship: 2

Relative features:
 - node2Voltage_delta_from_baseline
 - node2Voltage_pct_change_from_baseline
 - collectorEmitterCurrent_delta_from_baseline
 - collectorEmitterCurrent_pct_change_from_baseline
 - packageTemperature_delta_from_baseline
 - packageTemperature_pct_change_from_baseline

New relationship features:
 - current_per_node2V
 - node_voltage_drop


In [5]:
# ============================================================
# 3. HARD VALIDATION
# ============================================================

EXPECTED_RELATIVE = {
    "node2Voltage_delta_from_baseline",
    "node2Voltage_pct_change_from_baseline",
    "collectorEmitterCurrent_delta_from_baseline",
    "collectorEmitterCurrent_pct_change_from_baseline",
    "packageTemperature_delta_from_baseline",
    "packageTemperature_pct_change_from_baseline",
}

EXPECTED_RESIDUAL = {
    "current_residual",
    "current_residual_abs",
    "package_temperature_residual",
}

EXPECTED_NEW_RELATIONSHIP = {
    "current_per_node2V",
    "node_voltage_drop",
}

assert len(absolute_features) == 21, (
    f"Expected 21 absolute features, found {len(absolute_features)}"
)

assert set(relative_features) == EXPECTED_RELATIVE, (
    f"Relative feature mismatch:\n"
    f"missing = {EXPECTED_RELATIVE - set(relative_features)}\n"
    f"unexpected = {set(relative_features) - EXPECTED_RELATIVE}"
)

assert set(residual_features) == EXPECTED_RESIDUAL, (
    f"Residual feature mismatch:\n"
    f"{set(residual_features) ^ EXPECTED_RESIDUAL}"
)

assert set(relationship_features) == EXPECTED_NEW_RELATIONSHIP, (
    f"Relationship feature mismatch:\n"
    f"{set(relationship_features) ^ EXPECTED_NEW_RELATIONSHIP}"
)

MODEL_FEATURES = (
    absolute_features
    + relative_features
    + residual_features
    + relationship_features
)

assert len(MODEL_FEATURES) == 32
assert len(MODEL_FEATURES) == len(set(MODEL_FEATURES))

print("\n✅ FEATURE PLAN CONFIRMED")
print("21 absolute + 6 relative + 3 residual + 2 relationship = 32")


✅ FEATURE PLAN CONFIRMED
21 absolute + 6 relative + 3 residual + 2 relationship = 32


In [6]:
# ============================================================
# 4. MERGE BY STABLE WINDOW KEYS
# ============================================================

clean = absolute[
    META + absolute_features
].copy()

for df, cols, label in [
    (relative, relative_features, "relative"),
    (residual, residual_features, "residual"),
    (relationship, relationship_features, "relationship"),
]:

    merge_df = df[
        META + cols
    ].copy()

    clean = clean.merge(
        merge_df,
        on=META,
        how="left",
        validate="one_to_one",
    )

    print(
        f"After {label:12s} merge: {clean.shape}"
    )

After relative     merge: (687, 34)
After residual     merge: (687, 37)
After relationship merge: (687, 39)


In [7]:
# ============================================================
# 5. STRUCTURAL CHECKS
# ============================================================

assert len(clean) == 687

assert not clean.columns.duplicated().any()

missing_model_columns = [
    c for c in MODEL_FEATURES
    if c not in clean.columns
]

assert not missing_model_columns, (
    f"Missing model columns: {missing_model_columns}"
)

print("Rows:", len(clean))
print("Columns:", len(clean.columns))
print("Duplicate columns:", clean.columns[
    clean.columns.duplicated()
].tolist())

print("\n✅ Structural checks passed.")

Rows: 687
Columns: 39
Duplicate columns: []

✅ Structural checks passed.


In [8]:
# ============================================================
# 6. REFERENCE VARIABLES — KEPT OUTSIDE MODEL INPUT
# ============================================================

REFERENCE_COLUMNS = [
    "node2Voltage_baseline",
    "collectorEmitterCurrent_baseline",
    "packageTemperature_baseline",
    "expected_current",
    "expected_package_temperature",
]

reference = relative[META].copy()

for c in REFERENCE_COLUMNS:

    if c in relative.columns:
        reference[c] = relative[c].to_numpy()

    elif c in residual.columns:
        reference[c] = residual[c].to_numpy()

reference.to_csv(
    PROCESSED / "burninwatch_reference_features_final.csv",
    index=False
)

print("✅ Reference feature file saved.")

✅ Reference feature file saved.


In [9]:
# ============================================================
# 7. FEATURE QUALITY AUDIT
# ============================================================

audit_rows = []

for feature in MODEL_FEATURES:

    s = pd.to_numeric(
        clean[feature],
        errors="coerce"
    )

    audit_rows.append({
        "feature": feature,
        "missing_fraction": float(
            s.isna().mean()
        ),
        "finite_fraction": float(
            np.isfinite(s).mean()
        ),
        "nunique": int(
            s.nunique(dropna=True)
        ),
        "std": float(s.std()),
        "min": float(s.min()),
        "max": float(s.max()),
    })

audit = pd.DataFrame(audit_rows)

display(audit)

print("\nFeatures with missing values:")
display(
    audit[
        audit["missing_fraction"] > 0
    ]
)

,feature,missing_fraction,finite_fraction,nunique,std,min,max
0,supplyVoltage_mean,0.000000,1.000000,687,1.085914,2.446614,5.924302
1,supplyVoltage_std,0.000000,1.000000,687,0.103293,0.001044,1.541208
2,node2Voltage_mean,0.000000,1.000000,687,1.089040,1.333053,5.925280
3,node2Voltage_std,0.000000,1.000000,687,0.101803,0.001144,1.546131
4,node2Voltage_iqr,0.000000,1.000000,677,0.214785,0.001383,3.467087
5,node2Voltage_slope,0.000000,1.000000,687,0.017253,-0.225224,0.046391
6,collectorEmitterCurrent_mean,0.000000,1.000000,687,0.006106,0.083349,0.155985
7,collectorEmitterCurrent_std,0.000000,1.000000,687,0.010853,0.001843,0.079448
8,collectorEmitterCurrent_q05,0.000000,1.000000,58,0.004974,0.058853,0.144262
9,collectorEmitterCurrent_q95,0.000000,1.000000,104,0.012751,0.087395,0.199160



Features with missing values:


,feature,missing_fraction,finite_fraction,nunique,std,min,max
27,current_residual,0.005822,0.994178,683,0.008187,-0.030986,0.021212
28,current_residual_abs,0.005822,0.994178,683,0.005234,0.000004,0.030986
29,package_temperature_residual,0.005822,0.994178,683,107.268440,-448.052940,203.077978


In [10]:
# ============================================================
# 8. CORRELATION AUDIT
# ============================================================

corr = clean[
    MODEL_FEATURES
].corr()

upper = corr.where(
    np.triu(
        np.ones(corr.shape),
        k=1
    ).astype(bool)
)

high_corr = (
    upper
    .stack()
    .reset_index()
    .rename(columns={
        "level_0": "feature_1",
        "level_1": "feature_2",
        0: "correlation",
    })
)

high_corr["abs_correlation"] = (
    high_corr["correlation"].abs()
)

high_corr = high_corr.sort_values(
    "abs_correlation",
    ascending=False
)

display(
    high_corr[
        high_corr["abs_correlation"] >= 0.95
    ].head(30)
)

,feature_1,feature_2,correlation,abs_correlation
429,node1_node2_mean_diff,node_voltage_drop,0.999995,0.999995
460,collectorEmitterCurrent_delta_from_baseline,collectorEmitterCurrent_pct_change_from_baseline,0.998574,0.998574
307,packageTemperature_mean,packageTemperature_q05,0.998339,0.998339
308,packageTemperature_mean,packageTemperature_q95,0.998040,0.998040
441,node2Voltage_delta_from_baseline,node2Voltage_pct_change_from_baseline,0.997945,0.997945
79,node2Voltage_mean,node2Voltage_delta_from_baseline,0.997603,0.997603
1,supplyVoltage_mean,node2Voltage_mean,0.996766,0.996766
20,supplyVoltage_mean,node2Voltage_delta_from_baseline,0.995974,0.995974
318,packageTemperature_mean,packageTemperature_delta_from_baseline,0.995933,0.995933
369,packageTemperature_q95,packageTemperature_delta_from_baseline,0.994569,0.994569


In [11]:
# ============================================================
# 9. FEATURE MANIFEST
# ============================================================

def feature_family(name):

    if name in relative_features:
        return "relative_change"

    if name in residual_features:
        return "context_residual"

    if name in relationship_features:
        return "relationship"

    if "supplyVoltage" in name:
        return "operating_context"

    if (
        "node2Voltage" in name
        or "collectorEmitterCurrent" in name
    ):
        return "electrical_response"

    if "packageTemperature" in name:
        return "thermal_response"

    return "other"


manifest = pd.DataFrame({
    "feature_order": np.arange(
        1,
        len(MODEL_FEATURES) + 1
    ),
    "feature": MODEL_FEATURES,
})

manifest["family"] = (
    manifest["feature"]
    .apply(feature_family)
)

display(manifest)

print("\nFeature counts by family:")
display(
    manifest.groupby("family")
    .size()
    .reset_index(name="count")
)

,feature_order,feature,family
0,1,supplyVoltage_mean,operating_context
1,2,supplyVoltage_std,operating_context
2,3,node2Voltage_mean,electrical_response
3,4,node2Voltage_std,electrical_response
4,5,node2Voltage_iqr,electrical_response
5,6,node2Voltage_slope,electrical_response
6,7,collectorEmitterCurrent_mean,electrical_response
7,8,collectorEmitterCurrent_std,electrical_response
8,9,collectorEmitterCurrent_q05,electrical_response
9,10,collectorEmitterCurrent_q95,electrical_response



Feature counts by family:


,family,count
0,context_residual,3
1,electrical_response,10
2,operating_context,2
3,other,3
4,relationship,2
5,relative_change,6
6,thermal_response,6


In [12]:
# ============================================================
# 10. DEVICE / FILE DISTRIBUTION
# ============================================================

print("Device distribution:")

display(
    clean.groupby("device_id")
    .agg(
        windows=("window_index", "count"),
        files=("file_name", "nunique")
    )
    .reset_index()
)

print("\nFile distribution:")

display(
    clean.groupby(
        ["device_id", "file_name", "file_role"]
    )
    .size()
    .reset_index(name="windows")
)

Device distribution:


,device_id,windows,files
0,Device_2,175,3
1,Device_3,151,3
2,Device_4,254,2
3,Device_5,107,2



File distribution:


,device_id,file_name,file_role,windows
0,Device_2,Device2 1.mat,main,110
1,Device_2,Device2b 1.mat,b,64
2,Device_2,Device2check 1.mat,check,1
3,Device_3,Device3 1.mat,main,98
4,Device_3,Device3b 1.mat,b,52
5,Device_3,Device3check 1.mat,check,1
6,Device_4,Device4 1.mat,main,108
7,Device_4,Device4b 1.mat,b,146
8,Device_5,Device5 1.mat,main,105
9,Device_5,Device5check 1.mat,check,2


In [13]:
# ============================================================
# 11. SAVE FINAL MODELING DATASET
# ============================================================

final = clean.copy()

final.insert(
    0,
    "sample_id",
    np.arange(len(final))
)

FINAL_PATH = (
    PROCESSED /
    "burninwatch_modeling_dataset_final.csv"
)

AUDIT_PATH = (
    PROCESSED /
    "burninwatch_feature_audit_final.csv"
)

MANIFEST_PATH = (
    PROCESSED /
    "burninwatch_feature_manifest_final.csv"
)

final.to_csv(
    FINAL_PATH,
    index=False
)

audit.to_csv(
    AUDIT_PATH,
    index=False
)

manifest.to_csv(
    MANIFEST_PATH,
    index=False
)

print("Saved:")
print(" -", FINAL_PATH)
print(" -", AUDIT_PATH)
print(" -", MANIFEST_PATH)

Saved:
 - processed\burninwatch_modeling_dataset_final.csv
 - processed\burninwatch_feature_audit_final.csv
 - processed\burninwatch_feature_manifest_final.csv


In [14]:
# ============================================================
# 12. FINAL SANITY CHECK
# ============================================================

check = pd.read_csv(
    FINAL_PATH
)

model_columns = [
    c for c in check.columns
    if c in MODEL_FEATURES
]

relative_present = [
    c for c in EXPECTED_RELATIVE
    if c in check.columns
]

duplicates = (
    check.columns[
        check.columns.duplicated()
    ].tolist()
)

print("Final shape:", check.shape)
print("Rows:", len(check))
print("Model features:", len(model_columns))
print(
    "Relative features present:",
    len(relative_present)
)
print("Duplicate columns:", duplicates)

assert len(check) == 687
assert len(model_columns) == 32
assert len(relative_present) == 6
assert not duplicates

print("\n" + "=" * 70)
print("✅ BURNINWATCH FINAL DATASET READY FOR MODELING")
print("=" * 70)

Final shape: (687, 40)
Rows: 687
Model features: 32
Relative features present: 6
Duplicate columns: []

✅ BURNINWATCH FINAL DATASET READY FOR MODELING
